In [1]:
from rag_helper import RAGBase
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from ingest import load_faq_data
from rag_helper import RAGBase
import os

In [21]:
from groq import Groq
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"

In [4]:
import time
from sqlitesearch import TextSearchIndex

index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [120]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=3,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

Tool Calling

In [11]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query"
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}

In [54]:
system_instruction = [
    "Decide if search is needed before answering user queries.",
    "Use native 'search' tool when needed.",
    "If needed, call 'search' 1-5 times with short queries. If not, reply with the best answer you can provide."
]

messages = [
    {"role": "system", "content": "\n".join(system_instruction)},
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

In [36]:
response = groq_client.chat.completions.create(
    messages=messages,
    model=MODEL,
    tools=[search_tool],
)
response.choices[0].message

ChatCompletionMessage(content="I don't have access to information about a specific course. Can I help you search for course details?", role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=[ChatCompletionMessageToolCall(id='ayy0gnzvp', function=Function(arguments='{"query":"course details"}', name='search'), type='function')])

In [42]:
import json
call = response.choices[0].message.tool_calls[0].function
args = json.loads(call.arguments)

print("args: ", args)

result = search(**args)
result_json = json.dumps(result, indent=2)


args:  {'query': 'course details'}


In [55]:
messages.append(response.choices[0].message)
messages.append({
    "role": "tool",
    "tool_call_id": response.choices[0].message.tool_calls[0].id,
    "content": result_json,
})

In [56]:
response = groq_client.chat.completions.create(
    messages=messages,
    model=MODEL,
    tools=[search_tool],
)
response.choices[0].message

ChatCompletionMessage(content='Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None)

In [57]:
print(response.usage)

CompletionUsage(completion_tokens=25, prompt_tokens=1485, total_tokens=1510, completion_time=0.056589702, completion_tokens_details=None, prompt_time=0.05623553, prompt_tokens_details=None, queue_time=0.140305853, total_time=0.112825232)


**Agentic Loop**

In [124]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches when needed. If there is any potential typo, rewrite the queries to make them appropriate.

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [61]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question},
]

In [130]:
def make_call(tool_call):
    call = tool_call.function
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)
        
    result_json = json.dumps(result, indent=2)

    return {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result_json,
        "name": call.name
    }

In [131]:
def agent_loop(instructions, question, model=MODEL) -> str:
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ]
    it = 1

    while True:
        print("\n")
        print(f"iteration #{it}...")
        has_function_calls = False
        time.sleep(2)

        response = groq_client.chat.completions.create(
            messages=messages,
            model=model,
            tools=[search_tool],
            tool_choice="auto"
        )

        messages.append(response.choices[0].message)

        if not response.choices[0].message.tool_calls:
            print("ASSISTANT:")
            print(response.choices[0].message.content)
            last_answer = response.choices[0].message.content
        else:
            for tool_call in response.choices[0].message.tool_calls:
                print("function_call:", tool_call.function.name, tool_call.function.arguments)
                call_output = make_call(tool_call)
                print("function_output:", call_output)
                messages.append(call_output)
                has_function_calls = True

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [134]:
agent_loop(instructions, "How do I run Olama locally?", model="openai/gpt-oss-120b")



iteration #1...
function_call: search {"query":"Olama run locally"}
function_output: {'role': 'tool', 'tool_call_id': 'fc_d037cd1a-c1b5-4679-96fb-14b9233cd00b', 'content': '[\n  {\n    "id": "aa310de435",\n    "course": "llm-zoomcamp",\n    "section": "Module 1: RAG",\n    "question": "Can I run the course locally instead of Codespaces?",\n    "answer": "Yes. Codespaces is just the easiest way for everyone to start with the same environment.\\n\\nYou can run the course locally if you are comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module.\\n\\nIf you run locally, make sure you document your setup and keep your environment reproducible."\n  },\n  {\n    "id": "e8df9f0d12",\n    "course": "llm-zoomcamp",\n    "section": "Module 6: Best Practices",\n    "question": "Docker: When trying to run a streamlit app using docker-compose, I get: Error response from daemon: failed to create task for container: failed to create shim task: OCI runtime cr

'Here’s how you can get **Ollama** up and running on your own machine:\n\n1. **Install Ollama**  \n   - **macOS** – Download the `.pkg` installer from\u202f[https://ollama.com/download](https://ollama.com/download) and run it.  \n   - **Windows** – Download the `.msi` installer from the same page and follow the setup wizard.  \n   - **Linux** – Run the one‑liner in a terminal:  \n\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**  \n   After installation open a terminal and execute a model name, for example:\n\n   ```bash\n   ollama run llama3\n   ```\n\n   This will:\n   * download the LLaMA\u202f3 model (≈\u202f4\u202fGB) the first time,\n   * start the model as a local server, and\n   * give you a simple chat‑like prompt where you can type queries.\n\n3. **Verify the local server**  \n   Ollama runs an HTTP API on port\u202f11434. Test it with:\n\n   ```bash\n   curl http://localhost:11434\n   ```\n\n   You should see a JSON 

In [135]:
agent_loop(instructions, "what's queen gambit?")



iteration #1...
function_call: search {"query":"Queen's Gambit course"}
function_output: {'role': 'tool', 'tool_call_id': 'vx08k9ynm', 'content': '[\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n    "answer": "No, you can only get a certificate if you finish the course with a \\"live\\" cohort.\\n\\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\\n\\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."\n  },\n  {\n    "id": "bd31146b0e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "When will the course be offered next?",\n    "answer": "Summer 2027."\n  },\n  {\n    "id": "74eb249bbf",\n    "cour

"The search results did not provide any information about the Queen's Gambit. It's likely that the Queen's Gambit is not related to our course. \n\nAre there other areas that you want to explore?"

# ToyAIKIT

In [136]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [138]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

agent_tools = Tools()
agent_tools.add_tool(search)

In [139]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [140]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.